# TickVault: bounded streaming, exact queries, and exports

This tutorial runs offline by default. It creates **synthetic** BI5 ticks to demonstrate the same reader and storage APIs used with downloaded Dukascopy files. Synthetic ticks are not market observations. Install this checkout with `pip install -e .`, then select that Python environment in Jupyter.

The final optional download cell is disabled until you explicitly set `RUN_NETWORK = True`.

In [ ]:
from datetime import timedelta
from pathlib import Path
import sys
import tempfile

repo = Path.cwd()
if not (repo / 'examples' / 'offline_analysis.py').exists():
    repo = repo.parent
assert (repo / 'examples' / 'offline_analysis.py').exists(), 'Open this notebook from the TickVault checkout'
sys.path.insert(0, str(repo))

from examples.offline_analysis import create_demo_archive
from tick_vault import config, download_and_read
from tick_vault.reader import iter_tick_data, read_tick_data
from tick_vault.read.storage import export_sqlite, iter_sqlite, read_sqlite

original_config = config.CONFIG.model_dump()
workspace = tempfile.TemporaryDirectory(prefix='tickvault-notebook-')
base_directory = Path(workspace.name)
start, end = create_demo_archive(base_directory)
print('Synthetic archive:', base_directory)
print('UTC range:', start, end)

## Stream compressed hours in bounded batches

`iter_tick_data` emits at most `batch_size` rows per DataFrame. Thread and process backends bound their outstanding hourly decode jobs and preserve chronological output. A whole compressed hour is decoded at a time; the batch limit does not cap the memory needed for one hour. For a small range, `read_tick_data` materializes one DataFrame.

In [ ]:
rows = 0
for batch in iter_tick_data('EURUSD', start, end, batch_size=3, backend='thread', max_workers=2, show_progress=False):
    assert len(batch) <= 3
    rows += len(batch)
    print(batch.assign(spread=batch.ask - batch.bid))
assert rows == 8
print('Streamed ticks:', rows)

## Query exact UTC bounds inside an hour

Reader ranges are `[start, end)`: the start is included and the end is excluded. Naive input datetimes mean UTC; offset-aware datetimes are converted to UTC before hour selection. Returned DataFrame times are timezone-naive UTC for backward compatibility. Strict reads check attempted-hour coverage. Non-strict reads can skip never-downloaded hours, but corrupt or missing files marked as downloaded still fail. Ask and bid volumes retain TickVault's integer millionths convention: feed volumes multiplied by 1,000,000 and rounded.

In [ ]:
query_start = start + timedelta(minutes=15)
query_end = start + timedelta(minutes=45)
subset = read_tick_data('EURUSD', query_start, query_end, show_progress=False)
assert len(subset) == 2
assert (subset.time >= query_start.replace(tzinfo=None)).all() and (subset.time < query_end.replace(tzinfo=None)).all()
print(subset)

## Export and query SQLite without loading the full archive

Exports stream bounded reader batches into a local SQLite database. Query a smaller range with `read_sqlite`, or process query batches with `iter_sqlite`. HDF5 offers equivalent `export_hdf5`, `read_hdf5`, and `iter_hdf5` APIs after installing `tick-vault[hdf5]`. Replace is the default export policy; append does not remove duplicates from overlapping source ranges.

In [ ]:
database = base_directory / 'analysis.sqlite'
exported = export_sqlite(database, 'EURUSD', start, end, batch_size=3, show_progress=False)
assert exported == rows
sql_subset = read_sqlite(database, 'EURUSD', query_start, query_end)
assert len(sql_subset) == 2
assert sum(len(batch) for batch in iter_sqlite(database, 'EURUSD', batch_size=3)) == rows
print(sql_subset)
print('SQLite export:', database)

## Optional local HDF5 export

This cell runs when the optional PyTables dependency is installed (`pip install 'tick-vault[hdf5]'`). It uses only synthetic local data and otherwise skips cleanly.

In [ ]:
import importlib.util
if importlib.util.find_spec('tables') is not None:
    from tick_vault.read.storage import export_hdf5, iter_hdf5, read_hdf5
    hdf_file = base_directory / 'analysis.h5'
    assert export_hdf5(hdf_file, 'EURUSD', start, end, batch_size=3, show_progress=False) == rows
    assert len(read_hdf5(hdf_file, 'EURUSD', query_start, query_end)) == 2
    assert sum(len(batch) for batch in iter_hdf5(hdf_file, 'EURUSD', batch_size=3)) == rows
    print('HDF5 export:', hdf_file)
else:
    print('PyTables is not installed; optional HDF5 example skipped')

## Optional: download one historical hour

This cell makes no network request unless enabled. It uses a separate archive from the synthetic tutorial, one worker, and one historical hour. The convenience API downloads complete enclosing hours and reads the exact requested range. Omitted or future ends stop at the last complete UTC hour. In Jupyter, call async APIs with `await`; regular scripts use `asyncio.run(...)`.

Completed files remain resumable after interruption. A cooperative stop raises `CancelledError` from `download_and_read` instead of returning partial data as a completed result.

In [ ]:
RUN_NETWORK = False
if RUN_NETWORK:
    config.reload_config(base_directory=base_directory / 'real_feed', worker_per_proxy=1)
    network_data = await download_and_read('EURUSD', start, start + timedelta(hours=1), show_progress=False)
    print(network_data.head())
    print('Real feed rows:', len(network_data))
else:
    print('Live download disabled; no request sent')

## Clean up

The default tutorial archive is temporary. Copy any desired export before running this cleanup cell. Restoring the original configuration closes the temporary log before Windows removes the directory.

In [ ]:
config.reload_config(**original_config)
workspace.cleanup()
print('Temporary tutorial files removed')